# 01_dataclean_3_res_offshore.ipynb

Refactored short version using shared utility functions.


In [1]:
import pandas as pd
from cleaning_utils import run_source_pipeline, add_calendar_columns, aggregate_hourly
from verify_time_consistency import run_checks, run_hourly_checks


In [2]:
files = [
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201412312300-201512312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201512312300-201612312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201612312300-201712312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201712312300-201812312300 - 1.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201712312300-201812312300 - 2.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201812312300-201912312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_201912312300-202012312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_202012312300-202112312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_202112312300-202212312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_202212312300-202312312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_202312312300-202412312300.csv",
    "../../data/res/off_wind/GUI_WIND_SOLAR_GENERATION_FORECAST_OFFSHORE_202412312300-202512312300.csv",
]

KEEP_COLS = ['MTU (CET/CEST)', 'Day-ahead (MW)', 'Actual (MW)']
RENAME_MAP = {'MTU (CET/CEST)': 'period', 'Day-ahead (MW)': 'off_wind_da', 'Actual (MW)': 'off_wind_act'}
VALUE_COLS = ['off_wind_da', 'off_wind_act']
OUTPUT_CSV = '../../data_cleaned/by_source/03_RES_WIND_OFFSHORE.csv'
ROW_SELECTION_COL = None
ROW_SELECTION_DROP_VALUE = None


In [3]:
result = run_source_pipeline(
    files=files,
    keep_cols=KEEP_COLS,
    rename_map=RENAME_MAP,
    value_cols=VALUE_COLS,
    row_selection_col=ROW_SELECTION_COL,
    row_selection_drop_value=ROW_SELECTION_DROP_VALUE,
    include_calendar_columns=False,
)

raw_df = result.raw
df_utc_q = result.quarter_hour

# Calendar features are intentionally added here (deferred from source cleaning helper)
df_utc_q_cal = add_calendar_columns(df_utc_q.copy())
df_utc_h = aggregate_hourly(df_utc_q_cal, value_cols=VALUE_COLS)

raw_df.shape, df_utc_q.shape, df_utc_h.shape


Non-numeric values were found and coerced to NaN. Sample rows (with context):
                                   period    period_start_utc      period_end_utc off_wind_da invalid_column off_wind_act
01/10/2018 00:00:00 - 01/10/2018 00:15:00 2018-09-30 22:00:00 2018-09-30 22:15:00         n/e    off_wind_da          NaN
01/10/2018 00:15:00 - 01/10/2018 00:30:00 2018-09-30 22:15:00 2018-09-30 22:30:00         n/e    off_wind_da          NaN
01/10/2018 00:30:00 - 01/10/2018 00:45:00 2018-09-30 22:30:00 2018-09-30 22:45:00         n/e    off_wind_da          NaN
01/10/2018 00:45:00 - 01/10/2018 01:00:00 2018-09-30 22:45:00 2018-09-30 23:00:00         n/e    off_wind_da          NaN
01/10/2018 01:00:00 - 01/10/2018 01:15:00 2018-09-30 23:00:00 2018-09-30 23:15:00         n/e    off_wind_da          NaN
01/10/2018 01:15:00 - 01/10/2018 01:30:00 2018-09-30 23:15:00 2018-09-30 23:30:00         n/e    off_wind_da          NaN
01/10/2018 01:30:00 - 01/10/2018 01:45:00 2018-09-30 23:30:00 2018-0

((420768, 6), (420768, 4), (96432, 13))

In [4]:
# Consistency checks moved to separate script/module
# Quarter-hour consistency before calendar feature engineering
run_checks(df_utc_q, ts_col='period_start_utc')

# Hourly completeness after calendar engineering + mean aggregation
run_hourly_checks(df_utc_h, ts_col='period_start_utc')


--- Frequency summary ---
period_start_utc
0 days 00:00:00     35040
0 days 00:15:00    385727
Name: count, dtype: int64

--- 15-min completeness (expected 96/day) ---
is_complete
True     3651
False     368
Name: count, dtype: int64
Sample incomplete 15-min days:
                  n_periods  expected  is_complete
period_start_utc                                  
2014-12-31                4        96        False
2017-12-31              100        96        False
2018-01-01              192        96        False
2018-01-02              192        96        False
2018-01-03              192        96        False

--- Hourly completeness (expected 24/day) ---
is_complete
True     4017
False       2
Name: count, dtype: int64
Sample incomplete hourly days:
                  n_periods  expected  is_complete
period_start_utc                                  
2014-12-31                1        24        False
2025-12-31               23        24        False


In [5]:
df_utc_h.head()


,date,year,month,day,dayofyear,hour,week,dayofweek,off_wind_da,off_wind_act,period_start_utc,period_end_utc,c_by_hour
0,2014-12-31,2014,12,31,365,23,1,2,258.50,521.1775,2014-12-31 23:00:00,2015-01-01 00:00:00,4
1,2015-01-01,2015,1,1,1,0,1,3,258.75,520.8100,2015-01-01 00:00:00,2015-01-01 01:00:00,4
2,2015-01-01,2015,1,1,1,1,1,3,258.50,519.0325,2015-01-01 01:00:00,2015-01-01 02:00:00,4
3,2015-01-01,2015,1,1,1,2,1,3,259.00,522.5550,2015-01-01 02:00:00,2015-01-01 03:00:00,4
4,2015-01-01,2015,1,1,1,3,1,3,260.50,524.6200,2015-01-01 03:00:00,2015-01-01 04:00:00,4


In [6]:
df_utc_h.to_csv(OUTPUT_CSV, index=False)
print('saved:', OUTPUT_CSV)


saved: ../../data_cleaned/by_source/03_RES_WIND_OFFSHORE.csv
